# V70 RESUBMIT — Metric Fix Validation

**Objetivo**: testar se o V70 adapter atual (que deu 0.84 local) pode render **0.86-0.89** no Kaggle com a metric fix aplicada.

**Background**: Agent D1 descobriu que nosso `scripts/local_score.py` tinha 3 BUGS:
1. Strict bit regex `re.fullmatch(r'[01]+')` — metric oficial NÃO faz
2. `abs(a-b)<1e-2` em vez de `math.isclose(rel_tol=1e-2)`  — 1000x mais estrito
3. Faltava `enable_thinking=True` no chat template

**Agent D1 PART 2 simulação em 9500 rows**:
- V70 local old metric: 0.84
- V70 Kaggle real estimado (conservador): **0.86** (+2pp)
- V70 Kaggle real estimado (liberal): **0.89** (+5pp)

**Custo**: 1 Kaggle submit slot (5/dia)

**Riscos**:
- Adapter variance natural: ±0.01-0.02 entre runs
- Se ≥0.86: **confirma metric fix ajuda** + próximo passo V71.x
- Se ≥0.87: **JÁ SOMOS TOP 1**
- Se <0.84: variance regredir (2/5 submit slots reservados)

In [ ]:
# Cell 1 — Auto-discover V70 adapter in GDrive
# Procura automaticamente pelo SHA C4EA449A (V70 original que deu 0.84).
# Se nao encontrar, lista todos com ranking para escolha manual.

from google.colab import drive, userdata
drive.mount('/content/drive')

import hashlib
import json
from pathlib import Path

print('Searching all adapter_model.safetensors in GDrive...')
print('(This scans MyDrive recursively — may take 30-60s)')
print()

candidates = []
for p in Path('/content/drive/MyDrive').rglob('adapter_model.safetensors'):
    try:
        size_mb = p.stat().st_size / 1024**2
        # Compute SHA only for files <10GB (avoid reading massive files)
        if size_mb < 10000:
            h = hashlib.sha256(p.read_bytes()).hexdigest()[:8]
        else:
            h = 'TOO_BIG'

        # Check if adapter_config.json exists alongside
        cfg_path = p.parent / 'adapter_config.json'
        cfg = {}
        if cfg_path.exists():
            try:
                cfg = json.loads(cfg_path.read_text())
            except Exception:
                pass

        candidates.append({
            'path': str(p.parent),
            'sha8': h,
            'size_mb': size_mb,
            'r': cfg.get('r', '?'),
            'alpha': cfg.get('lora_alpha', '?'),
            'modified': p.stat().st_mtime,
        })
    except Exception as e:
        print(f'  Skipped {p}: {e}')

# Sort: C4EA449A first, then smaller size (true LoRA), then newest
def sort_key(c):
    is_v70 = 0 if c['sha8'].lower() == 'c4ea449a' else 1
    is_small = c['size_mb']  # smaller = more likely pure LoRA
    return (is_v70, is_small)

candidates.sort(key=sort_key)

print(f'{"Rank":<6}{"SHA8":<10}{"Size MB":<12}{"r":<6}{"alpha":<8}{"Path"}')
print('-' * 120)
for i, c in enumerate(candidates):
    marker = ' <-- V70!' if c['sha8'].lower() == 'c4ea449a' else ''
    print(f'{i:<6}{c["sha8"]:<10}{c["size_mb"]:<12.1f}{str(c["r"]):<6}{str(c["alpha"]):<8}{c["path"]}{marker}')

# Decide V70_PATH
V70_PATH = None
v70_exact = [c for c in candidates if c['sha8'].lower() == 'c4ea449a']
if v70_exact:
    V70_PATH = v70_exact[0]['path']
    print(f'\nCONFIRMED: V70 (C4EA449A) found at: {V70_PATH}')
else:
    print('\nWARNING: SHA C4EA449A not found in GDrive.')
    print('This means the original V70 adapter (that scored 0.84) may be lost/elsewhere.')
    print()
    # Suggest smallest reasonable LoRA (<500MB, r=32)
    valid_lora = [c for c in candidates if 50 < c['size_mb'] < 500 and c['r'] == 32]
    if valid_lora:
        V70_PATH = valid_lora[0]['path']
        print(f'FALLBACK SUGGESTION (smallest r=32 LoRA): {V70_PATH}')
        print('  SHA:', valid_lora[0]['sha8'], '| Size:', f'{valid_lora[0]["size_mb"]:.1f} MB')
        print('  NOT confirmed V70. Submit with caution.')
    elif candidates:
        print('No small LoRA found. All candidates are too large or different config.')
        print(f'Largest candidate (may have lm_head bundled): {candidates[0]["path"]}')
        print('Recommendation: ABORT this notebook, go to PASSO 2 (V70.5 train from scratch)')

assert V70_PATH is not None, 'No usable adapter found. Go to PASSO 2 (V70.5 notebook)'
print(f'\nV70_PATH = {V70_PATH}')


In [ ]:
# Cell 2 — Cross-reference with Kaggle submission history
# Descobrir qual adapter deu 0.84 no Kaggle via API de submissoes.
# Se encontrar submission 0.84, podemos correlacionar com arquivo no GDrive.

import os, subprocess, csv, io

# Kaggle creds
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME') or 'felipe1983'
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY') or ''

if not os.environ['KAGGLE_KEY']:
    print('WARNING: KAGGLE_KEY not in Colab secrets. Skipping Kaggle API query.')
    print('Add KAGGLE_KEY in Colab -> Secrets (left panel)')
else:
    # Install kaggle CLI if missing
    subprocess.run(['pip', 'install', '-q', 'kaggle'], check=False)

    # Query submissions
    result = subprocess.run(
        ['kaggle', 'competitions', 'submissions',
         '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'],
        capture_output=True, text=True, timeout=60,
    )

    if result.returncode == 0 and result.stdout:
        reader = csv.DictReader(io.StringIO(result.stdout))
        subs = list(reader)
        print(f'Total submissions: {len(subs)}')
        print()
        print(f'{"Date":<22}{"Status":<12}{"Public Score":<14}{"Description"[:40]:<40}')
        print('-' * 90)

        best_score = 0
        best_sub = None
        for s in subs[:20]:  # Top 20 mais recentes
            score_str = s.get('publicScore', '')
            try:
                score = float(score_str)
            except (ValueError, TypeError):
                score = 0
            date = s.get('date', '')[:19]
            status = s.get('status', '')[:10]
            desc = s.get('description', '')[:40]
            marker = ' <- BEST' if score > best_score else ''
            if score > best_score and score > 0:
                best_score = score
                best_sub = s
                marker = ' <- NEW BEST'
            print(f'{date:<22}{status:<12}{score_str:<14}{desc:<40}{marker}')

        if best_sub:
            print()
            print(f'BEST submission: score={best_score}')
            print(f'  Description: {best_sub.get("description")}')
            print(f'  File: {best_sub.get("fileName", "?")}')
            print(f'  Date: {best_sub.get("date")}')
        else:
            print('\nNo scored submissions found.')
    else:
        print('Kaggle API error:', result.stderr[:300])

# Decision based on Cell 1 discovery + Kaggle history:
print()
print('=' * 70)
print('DECISION TREE:')
print('=' * 70)
print(f'Adapter selected in Cell 1: {V70_PATH}')
print()
print('If Cell 1 found C4EA449A: proceed to Cell 3 (clone repo + eval)')
print('If Cell 1 found fallback (not V70): consider aborting this notebook')
print('If Cell 1 found NO usable adapter: STOP, go to PASSO 2 (KG1_V70_5_FIXED_METRIC.ipynb)')


In [ ]:
# Cell 3 — Clone KG1 repo WITH metric-fixed scripts
# Now that we pushed to GitHub, clone directly.

import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/FELIPEACASTRO/KG1-NVIDIA.git'
BRANCH = 'claude/competent-shamir'
REPO_PATH = '/content/kg1'

if not Path(REPO_PATH).exists():
    print(f'Cloning {REPO_URL} branch {BRANCH}...')
    result = subprocess.run(
        ['git', 'clone', '--depth=1', '-b', BRANCH, REPO_URL, REPO_PATH],
        capture_output=True, text=True, timeout=120,
    )
    print(result.stdout[-1000:])
    if result.returncode != 0:
        print('Clone failed:', result.stderr[-1000:])
        raise RuntimeError('Cannot clone repo — check network or branch name')
else:
    print(f'Repo already exists at {REPO_PATH}, pulling latest...')
    subprocess.run(['git', '-C', REPO_PATH, 'pull'], capture_output=True, text=True)

# Verify critical scripts exist
critical = [
    'scripts/local_score.py',
    'scripts/kg1_local_metric_gate.py',
    'scripts/kg1_submission_gate.py',
    'scripts/submit_kaggle.py',
]
for p in critical:
    full = Path(REPO_PATH) / p
    if full.exists():
        print(f'  OK {p} ({full.stat().st_size} bytes)')
    else:
        print(f'  MISSING {p}')

sys.path.insert(0, REPO_PATH)
print(f'\nRepo ready at {REPO_PATH}')
print('Scripts above have METRIC FIXES applied (D1 reverse engineering 2026-04-21)')


In [ ]:
# Cell 4 — Local gate with CORRECTED metric (D1 fix)
# This tests if V70 passes our CORRECTED local eval
# Expected: higher score than old buggy metric

import subprocess, pandas as pd

result = subprocess.run([
    'python', '/content/kg1/scripts/local_score.py',
    '--adapter', V70_PATH,
    '--n-samples', '600',
    '--output-csv', '/content/v70_local_eval_metric_fixed.csv',
], capture_output=True, text=True, timeout=3600)

print('STDOUT (last 3000 chars):')
print(result.stdout[-3000:])
print('\nSTDERR (last 1000 chars):')
print(result.stderr[-1000:])

# Load results
try:
    df = pd.read_csv('/content/v70_local_eval_metric_fixed.csv')
    overall = df['correct'].mean()
    print(f'\n🎯 V70 local eval with METRIC FIXED: {overall:.4f}')
    print('(Compare to old buggy local: 0.84)')
    print()
    if 'category' in df.columns:
        print('Per category:')
        print(df.groupby('category')['correct'].mean().sort_values(ascending=False))
except Exception as e:
    print(f'Eval failed: {e}')

In [ ]:
# Cell 5 — Prepare submission ZIP (V70 unchanged, just repackage)
import shutil, zipfile
from pathlib import Path

submission_dir = Path('/content/v70_submission')
submission_dir.mkdir(exist_ok=True)

# Copy adapter files
shutil.copy(V70_PATH + '/adapter_config.json', submission_dir / 'adapter_config.json')
shutil.copy(V70_PATH + '/adapter_model.safetensors', submission_dir / 'adapter_model.safetensors')

# Validate
cfg = json.loads((submission_dir / 'adapter_config.json').read_text())
assert cfg.get('r') == 32, f'r must be 32, got {cfg.get("r")}'
assert 'in_proj' in cfg.get('target_modules', []), 'in_proj required'

# Zip
zip_path = Path('/content/v70_resubmit.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in submission_dir.glob('*'):
        zf.write(f, arcname=f.name)

print(f'ZIP created: {zip_path}')
print(f'Size: {zip_path.stat().st_size / 1024**2:.1f} MB')
print(f'Contents: {[n.filename for n in zipfile.ZipFile(zip_path).infolist()]}')

In [ ]:
# Cell 6 — Kaggle submission gate (sanity check)
import subprocess

result = subprocess.run([
    'python', '/content/kg1/scripts/kg1_submission_gate.py',
    '--adapter-zip', '/content/v70_resubmit.zip',
    '--test-csv', '/content/kg1/data/kaggle/unzipped/test.csv',  # or Kaggle input path
    '--fail-on-block',
], capture_output=True, text=True)

print(result.stdout[-2000:])
print('---STDERR---')
print(result.stderr[-500:])

assert result.returncode == 0, 'Gate BLOCKED — do not submit'
print('\n✅ Gate PASS — safe to submit')

In [ ]:
# Cell 7 — Submit to Kaggle (CONSUMES 1/5 daily slot)
import subprocess, os

# Kaggle creds
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME') or 'felipe1983'
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
assert os.environ['KAGGLE_KEY'], 'KAGGLE_KEY required in Colab secrets'

# Final safety check — user confirmation
CONFIRM = input('Confirm submit V70 to Kaggle? (yes/no): ').strip().lower()
assert CONFIRM == 'yes', 'Aborted by user'

result = subprocess.run([
    'kaggle', 'competitions', 'submit',
    '-c', 'nvidia-nemotron-model-reasoning-challenge',
    '-f', '/content/v70_resubmit.zip',
    '-m', 'V70 resubmit — metric correction D1 test (2026-04-21)',
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

# Wait ~3 min for scoring
print('\n⏳ Kaggle scoring takes ~5-15 min. Check LB:')
print('  kaggle competitions submissions -c nvidia-nemotron-model-reasoning-challenge')

In [ ]:
# Cell 8 — Monitor submission status
import subprocess, time

for attempt in range(20):
    time.sleep(60)  # wait 60s
    result = subprocess.run([
        'kaggle', 'competitions', 'submissions',
        '-c', 'nvidia-nemotron-model-reasoning-challenge',
        '--csv',
    ], capture_output=True, text=True)
    
    # Parse CSV
    import csv, io
    reader = csv.DictReader(io.StringIO(result.stdout))
    latest = next(reader, None)
    if latest:
        status = latest.get('status', '?')
        score = latest.get('publicScore', '?')
        print(f'[{attempt+1}/20] status={status} score={score}')
        if status.lower() in ['complete', 'failed', 'error']:
            print(f'\n🎯 FINAL STATUS: {status}')
            print(f'🎯 SCORE: {score}')
            if score != '?' and float(score) >= 0.87:
                print('🏆 TOP 1 CONFIRMED!')
            elif score != '?' and float(score) >= 0.86:
                print('✅ Plateau atingido! Continue V71+ roadmap')
            elif score != '?' and float(score) >= 0.84:
                print('⚖️ No regresssion. Metric fix impact menor que simulação.')
            else:
                print('⚠️ REGRESSION — review submission or variance')
            break
else:
    print('⏱️ Timeout — check manually')

## 🎯 Next Steps Based on Outcome

### If score ≥ 0.87 (TOP 1 achieved!)
1. **Celebrate** 🎉
2. Monitor LB for 48h — confirm ranking holds
3. Prepare writeup required for prize eligibility
4. Optional: continue V71+ for margin

### If score ≥ 0.86 (plateau reached)
1. Gap 0.86→0.87 = 0.01pp
2. V71.2 (bit pairs) + V71.3 (cryptarithm) = **+0.010-0.020** → should break plateau
3. Run `scripts/v71_prepare_training_data.py --include-programmatic` locally
4. Train V71 notebook — see `KG1_V70_5_FIXED_METRIC.ipynb`

### If score 0.84-0.86 (moderate impact)
- Metric fix confirmed working but model needs improvements
- Proceed with V71+ full roadmap (V70.5 → V71.x → V72 → V72.5)

### If score < 0.84 (regression)
- Natural variance (±0.02 known)
- Retry once (1 more slot)
- Se persistir: investigar submission integrity (zip correto, adapter config, etc)
